In [1]:
%run /home/hadoop/agrim_cdp/common_utils/db.ipynb

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import logging
from datetime import datetime
import boto3
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, StructType
from pyspark.sql.functions import col, current_timestamp, from_unixtime, to_date, lit, explode, to_json

In [3]:
# ---- Configurations ----
BUCKET = 'agrim-cdp'
PREFIX = 'data-landing/webengage/webengage_events_raw/'
DEST_PREFIX = 'data-processing/webengage/webengage_events_raw/'
ARCH_PREFIX = 'data-archive/webengage/webengage_events_raw/'
BATCH_SIZE = 1000
TABLE_NAME = 'cdp_raw_db.webengage_events_raw'
S3_REGION = 'ap-south-1'
RUN_DATE = datetime.today().strftime('%Y-%m-%d')

In [4]:
logger = logging.getLogger('WEBENGAGE_DATA_LOAD')
logging.basicConfig(level=logging.INFO)

# Create console handler
handler = logging.StreamHandler()

# Add formatter with timestamp
formatter = logging.Formatter(
    fmt='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
handler.setFormatter(formatter)

# Add handler to logger
logger.addHandler(handler)

In [5]:
import os
os.environ["SPARK_HOME"] = "/home/hadoop/.local/lib/python3.9/site-packages/pyspark"  # Or wherever your Spark is
os.environ["PATH"] = os.environ["SPARK_HOME"] + "/bin:" + os.environ["PATH"]

In [6]:
spark = SparkSession.builder \
    .appName("webengage_batch_load") \
    .config("spark.jars", "/home/hadoop/agrim_cdp/common_jars/postgresql-42.2.24.jar") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()
s3 = boto3.client('s3', region_name=S3_REGION)

:: loading settings :: url = jar:file:/home/hadoop/.local/lib/python3.9/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hadoop/.ivy2/cache
The jars for the packages stored in: /home/hadoop/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-5315eb92-0b7e-4e2f-83ac-50de104f8ce8;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in spark-list


	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in spark-list
:: resolution report :: resolve 270ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from spark-list in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from spark-list in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-5315eb92-0b7e-4e2f-83ac-50de104f8ce8
	confs: [default]
	0 artifacts copied, 3 already retrieved (0k

25/07/09 08:45:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


25/07/09 08:45:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/07/09 08:45:10 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/07/09 08:45:10 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


INFO:botocore.credentials:Found credentials from IAM Role: AmazonEMR-InstanceProfile-20250415T154408


In [7]:
spark._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

In [8]:
response = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=BATCH_SIZE)
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key']]

In [9]:
#Copy each file with new key
for key in files:
    # Strip the prefix to get the relative path
    relative_path = key[len(PREFIX):]
    if relative_path:
        new_key = DEST_PREFIX + relative_path + '.gz'
        arch_key = ARCH_PREFIX + 'run_date=' + RUN_DATE + '/' + relative_path + '.gz'
    
        # Move the files to processing folder
        s3.copy_object(
            Bucket=BUCKET,
            CopySource={'Bucket': BUCKET, 'Key': key},
            Key=new_key
        )

        # Archive the Files
        s3.copy_object(
            Bucket=BUCKET,
            CopySource={'Bucket': BUCKET, 'Key': key},
            Key=arch_key
        )

logger.info("All files copied with .gz extension.")

2025-07-09 08:45:14 - INFO - All files copied with .gz extension.


INFO:WEBENGAGE_DATA_LOAD:All files copied with .gz extension.


In [10]:
webengage_schema = StructType([
    StructField("category", StringType(), True),
    StructField("cuid", StringType(), True),
    StructField("event_data", StructType([
        StructField("Business_name", StringType(), True),
        StructField("Comment", StringType(), True),
        StructField("Name", StringType(), True),
        StructField("address", StringType(), True),
        StructField("amplified", StringType(), True),
        StructField("app_version_code_new", StringType(), True),
        StructField("app_version_code_old", StringType(), True),
        StructField("bill_book_upload", StringType(), True),
        StructField("bucket_value", StringType(), True),
        StructField("cart_details", ArrayType(
            StructType([
                StructField("price", StringType(), True),
                StructField("product_id", StringType(), True),
                StructField("product_name", StringType(), True),
                StructField("quantity", StringType(), True),
                StructField("variation_id", StringType(), True)
            ])
        ), True),
        StructField("category", StringType(), True),
        StructField("category_name", StringType(), True),
        StructField("city", StringType(), True),
        StructField("coupon_code", StringType(), True),
        StructField("coupon_status", StringType(), True),
        StructField("currency", StringType(), True),
        StructField("discount_amount", StringType(), True),
        StructField("error_code", StringType(), True),
        StructField("error_reason", StringType(), True),
        StructField("is_masked", StringType(), True),
        StructField("login_method", StringType(), True),
        StructField("messageId", StringType(), True),
        StructField("method", StringType(), True),
        StructField("order_id", StringType(), True),
        StructField("pancard_status", StringType(), True),
        StructField("phone_number", StringType(), True),
        StructField("pin_code", StringType(), True),
        StructField("price", StringType(), True),
        StructField("product_details", ArrayType(
            StructType([
                StructField("product_id", StringType(), True),
                StructField("product_name", StringType(), True),
                StructField("quantity", StringType(), True),
                StructField("variation_id", StringType(), True)
            ])
        ), True),
        StructField("product_id", StringType(), True),
        StructField("product_name", StringType(), True),
        StructField("provider", StringType(), True),
        StructField("quantity", StringType(), True),
        StructField("reason", StringType(), True),
        StructField("registration_type", StringType(), True),
        StructField("result", StringType(), True),
        StructField("result_count", StringType(), True),
        StructField("sdk_meta", StringType(), True),
        StructField("search_query", StringType(), True),
        StructField("source", StringType(), True),
        StructField("state", StringType(), True),
        StructField("to", StringType(), True),
        StructField("total_items", StringType(), True),
        StructField("total_price", StringType(), True),
        StructField("total_quantity", StringType(), True),
        StructField("total_value", StringType(), True),
        StructField("tracker_id", StringType(), True),
        StructField("url", StringType(), True),
        StructField("user_id", StringType(), True),
        StructField("value", StringType(), True),
        StructField("variation_id", StringType(), True),
        StructField("wsp_name", StringType(), True)
    ])),
    StructField("event_name", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("license_code", StringType(), True),
    StructField("luid", StringType(), True),
    StructField("system_data", StructType([
        StructField("Advertising ID", StringType(), True),
        StructField("App ID", StringType(), True),
        StructField("App Version", StringType(), True),
        StructField("App Version Code", StringType(), True),
        StructField("Brand", StringType(), True),
        StructField("Browser Name", StringType(), True),
        StructField("Browser Version", StringType(), True),
        StructField("Campaign Id", StringType(), True),
        StructField("Carrier", StringType(), True),
        StructField("City", StringType(), True),
        StructField("Country", StringType(), True),
        StructField("Device", StringType(), True),
        StructField("Device Manufacturer", StringType(), True),
        StructField("Device Model", StringType(), True),
        StructField("Device Type", StringType(), True),
        StructField("Journey ID", StringType(), True),
        StructField("Locale", StringType(), True),
        StructField("OS Name", StringType(), True),
        StructField("OS version", StringType(), True),
        StructField("Platform", StringType(), True),
        StructField("SDK Version", StringType(), True),
        StructField("UTM Medium", StringType(), True),
        StructField("UTM Name", StringType(), True),
        StructField("UTM Source", StringType(), True),
        StructField("Variation Id", StringType(), True)
    ]))
])

In [11]:
df = spark.read.schema(webengage_schema).json("s3a://agrim-cdp/data-processing/webengage/webengage_events_raw/")

25/07/09 08:45:16 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


25/07/09 08:45:17 WARN CredentialsLegacyConfigLocationProvider: Found the legacy config profiles file at [/home/hadoop/.aws/config]. Please move it to the latest default location [~/.aws/credentials].


25/07/09 08:45:20 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [12]:
# Flatten nested fields
df = df.select(
    col("category"),
    col("cuid"),
    col("event_data.Business_name").alias("event_data_business_name"),
    col("event_data.Comment").alias("event_data_comment"),
    col("event_data.Name").alias("event_data_name"),
    col("event_data.address").alias("event_data_address"),
    col("event_data.amplified").alias("event_data_amplified"),
    col("event_data.app_version_code_new").alias("event_data_app_version_code_new"),
    col("event_data.app_version_code_old").alias("event_data_app_version_code_old"),
    col("event_data.bill_book_upload").alias("event_data_bill_book_upload"),
    col("event_data.bucket_value").alias("event_data_bucket_value"),
    col("event_data.cart_details").alias("event_data_cart_details"),
    col("event_data.category").alias("event_data_category"),
    col("event_data.category_name").alias("event_data_category_name"),
    col("event_data.city").alias("event_data_city"),
    col("event_data.coupon_code").alias("event_data_coupon_code"),
    col("event_data.coupon_status").alias("event_data_coupon_status"),
    col("event_data.currency").alias("event_data_currency"),
    col("event_data.discount_amount").alias("event_data_discount_amount"),
    col("event_data.error_code").alias("event_data_error_code"),
    col("event_data.error_reason").alias("event_data_error_reason"),
    col("event_data.is_masked").alias("event_data_is_masked"),
    col("event_data.login_method").alias("event_data_login_method"),
    col("event_data.messageId").alias("event_data_messageid"),
    col("event_data.method").alias("event_data_method"),
    col("event_data.order_id").alias("event_data_order_id"),
    col("event_data.pancard_status").alias("event_data_pancard_status"),
    col("event_data.phone_number").alias("event_data_phone_number"),
    col("event_data.pin_code").alias("event_data_pin_code"),
    col("event_data.price").alias("event_data_price"),
    col("event_data.product_details").alias("event_data_product_details"),
    col("event_data.product_id").alias("event_data_product_id"),
    col("event_data.product_name").alias("event_data_product_name"),
    col("event_data.provider").alias("event_data_provider"),
    col("event_data.quantity").alias("event_data_quantity"),
    col("event_data.reason").alias("event_data_reason"),
    col("event_data.registration_type").alias("event_data_registration_type"),
    col("event_data.result").alias("event_data_result"),
    col("event_data.result_count").alias("event_data_result_count"),
    col("event_data.sdk_meta").alias("event_data_sdk_meta"),
    col("event_data.search_query").alias("event_data_search_query"),
    col("event_data.source").alias("event_data_source"),
    col("event_data.state").alias("event_data_state"),
    col("event_data.to").alias("event_data_to"),
    col("event_data.total_items").alias("event_data_total_items"),
    col("event_data.total_price").alias("event_data_total_price"),
    col("event_data.total_quantity").alias("event_data_total_quantity"),
    col("event_data.total_value").alias("event_data_total_value"),
    col("event_data.tracker_id").alias("event_data_tracker_id"),
    col("event_data.url").alias("event_data_url"),
    col("event_data.user_id").alias("event_data_user_id"),
    col("event_data.value").alias("event_data_value"),
    col("event_data.variation_id").alias("event_data_variation_id"),
    col("event_data.wsp_name").alias("event_data_wsp_name"),
    col("event_name"),
    col("event_time"),
    col("license_code"),
    col("luid"),
    col("system_data.Advertising ID").alias("system_data_advertising_id"),
    col("system_data.App ID").alias("system_data_app_id"),
    col("system_data.App Version").alias("system_data_app_version"),
    col("system_data.App Version Code").alias("system_data_app_version_code"),
    col("system_data.Brand").alias("system_data_brand"),
    col("system_data.Browser Name").alias("system_data_browser_name"),
    col("system_data.Browser Version").alias("system_data_browser_version"),
    col("system_data.Campaign Id").alias("system_data_campaign_id"),
    col("system_data.Carrier").alias("system_data_carrier"),
    col("system_data.City").alias("system_data_city"),
    col("system_data.Country").alias("system_data_country"),
    col("system_data.Device").alias("system_data_device"),
    col("system_data.Device Manufacturer").alias("system_data_device_manufacturer"),
    col("system_data.Device Model").alias("system_data_device_model"),
    col("system_data.Device Type").alias("system_data_device_type"),
    col("system_data.Journey ID").alias("system_data_journey_id"),
    col("system_data.Locale").alias("system_data_locale"),
    col("system_data.OS Name").alias("system_data_os_name"),
    col("system_data.OS version").alias("system_data_os_version"),
    col("system_data.Platform").alias("system_data_platform"),
    col("system_data.SDK Version").alias("system_data_sdk_version"),
    col("system_data.UTM Medium").alias("system_data_utm_medium"),
    col("system_data.UTM Name").alias("system_data_utm_name"),
    col("system_data.UTM Source").alias("system_data_utm_source"),
    col("system_data.Variation Id").alias("system_data_variation_id")
)

# Add derived timestamp columns
df = df.withColumn("event_time_timestamp", from_unixtime((col("event_time") / 1000).cast("long")))
df = df.withColumn("event_time_date", to_date(from_unixtime((col("event_time") / 1000).cast("long"))))
df = df.withColumn("create_timestamp", current_timestamp())

df = df.withColumn("event_data_cart_details", to_json(col("event_data_cart_details")))
df = df.withColumn("event_data_product_details", to_json(col("event_data_product_details")))

In [13]:
postgres_columns = [
    'category', 'cuid',
    'event_data_business_name', 'event_data_comment', 'event_data_name', 'event_data_address', 'event_data_amplified',
    'event_data_app_version_code_new', 'event_data_app_version_code_old', 'event_data_bill_book_upload',
    'event_data_bucket_value', 'event_data_cart_details', 'event_data_category', 'event_data_category_name',
    'event_data_city', 'event_data_coupon_code', 'event_data_coupon_status', 'event_data_currency',
    'event_data_discount_amount', 'event_data_error_code', 'event_data_error_reason', 'event_data_is_masked',
    'event_data_login_method', 'event_data_messageid', 'event_data_method', 'event_data_order_id',
    'event_data_pancard_status', 'event_data_phone_number', 'event_data_pin_code', 'event_data_price',
    'event_data_product_details', 'event_data_product_id', 'event_data_product_name', 'event_data_provider',
    'event_data_quantity', 'event_data_reason', 'event_data_registration_type', 'event_data_result',
    'event_data_result_count', 'event_data_sdk_meta', 'event_data_search_query', 'event_data_source',
    'event_data_state', 'event_data_to', 'event_data_tonumber', 'event_data_total_items', 'event_data_total_price',
    'event_data_total_quantity', 'event_data_total_value', 'event_data_tracker_id', 'event_data_url',
    'event_data_user_id', 'event_data_value', 'event_data_variation_id', 'event_data_wsp_name',
    'event_name', 'event_time', 'license_code', 'luid',
    'system_data_advertising_id', 'system_data_app_id', 'system_data_app_version', 'system_data_app_version_code',
    'system_data_brand', 'system_data_browser_name', 'system_data_browser_version', 'system_data_campaign_id',
    'system_data_carrier', 'system_data_city', 'system_data_country', 'system_data_device',
    'system_data_device_manufacturer', 'system_data_device_model', 'system_data_device_type',
    'system_data_journey_id', 'system_data_locale', 'system_data_os_name', 'system_data_os_version',
    'system_data_platform', 'system_data_sdk_version', 'system_data_utm_medium', 'system_data_utm_name',
    'system_data_utm_source', 'system_data_variation_id',
    'event_time_timestamp', 'event_time_date', 'create_timestamp'
]

In [14]:
# Add missing columns with lit(None) and reorder:
for col_name in postgres_columns:
    if col_name not in df.columns:
        df = df.withColumn(col_name, lit(None).cast("string"))

df = df.select(postgres_columns)

In [15]:
df.count()

1168

In [16]:
RDS_HOST, RDS_DBNM, RDS_USER, RDS_PASSWORD, RDS_PORT = get_rds_credentials()

In [17]:
# ---- Step 4: Write to RDS using JDBC ----
df.write \
    .format("jdbc") \
    .option("url", f'jdbc:postgresql://{RDS_HOST}:{RDS_PORT}/{RDS_DBNM}') \
    .option("dbtable", TABLE_NAME) \
    .option("user", RDS_USER) \
    .option("password", RDS_PASSWORD) \
    .option("driver", "org.postgresql.Driver") \
    .mode("append") \
    .save()

In [18]:
# Delete the files from source and processing layer
for key in files:
    relative_path = key[len(PREFIX):]
    if relative_path:
        new_key = DEST_PREFIX + relative_path + '.gz'
        try:
            s3.delete_object(Bucket=BUCKET, Key=key)
            s3.delete_object(Bucket=BUCKET, Key=new_key)
        except Exception as e:
            logger.warning(f"Failed to delete {key}: {str(e)}")

In [19]:
logger.info('WebEngage Data Load Completed')

2025-07-09 08:45:28 - INFO - WebEngage Data Load Completed


INFO:WEBENGAGE_DATA_LOAD:WebEngage Data Load Completed
